# 🚀 Image → 3D in Colab (TripoSR — Fast Demo)

Upload **any image** → get a downloadable `.glb` 3D model in under 2 seconds.

**Runtime:** GPU (T4 free tier is fine)  
**VRAM needed:** ~5 GB  

▶️ **Runtime → Change runtime type → T4 GPU → Save**

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Enable GPU in Runtime → Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
!pip install -q trimesh rembg huggingface-hub
import tsr; print('TripoSR installed OK')

In [ ]:
# ── 2. Load model (downloads ~1 GB weights once, cached after that) ───────
import torch
from tsr.system import TSR

print('Loading model …')
model = TSR.from_pretrained(
    'stabilityai/TripoSR',
    config_name='config.yaml',
    weight_name='model.ckpt',
)
model.renderer.set_chunk_size(8192)
model = model.to('cuda')
print('Model loaded!')

In [ ]:
# ── 3. Upload your image ──────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
print(f'Uploaded: {img_path}')

In [ ]:
# ── 4. Remove background ──────────────────────────────────────────────────
import rembg
from PIL import Image
import numpy as np

session = rembg.new_session('birefnet-general')
image = Image.open(img_path).convert('RGBA')
image_no_bg = rembg.remove(image, session=session)

# Composite on white background
white = Image.new('RGBA', image_no_bg.size, 'WHITE')
white.paste(image_no_bg, (0, 0), image_no_bg)
image_clean = white.convert('RGB').resize((512, 512), Image.LANCZOS)

image_clean.save('/content/input_clean.png')
display(image_clean)
print('Background removed ✓')

In [ ]:
# ── 5. Run TripoSR ────────────────────────────────────────────────────────
import time, trimesh

t0 = time.time()
with torch.no_grad():
    scene_codes = model([image_clean], device='cuda')
    meshes = model.extract_mesh(scene_codes, resolution=256, threshold=25.0)
latency = time.time() - t0

mesh = meshes[0]
print(f'Done in {latency:.2f}s — {len(mesh.vertices)} vertices, {len(mesh.faces)} faces')

In [ ]:
# ── 6. Export to GLB and download ─────────────────────────────────────────
out_path = '/content/output.glb'
mesh.export(out_path)
print(f'Saved to {out_path}')

files.download(out_path)
print('⬇️  Download started!')

In [ ]:
# ── 7. (Optional) Quick 360° preview in notebook ──────────────────────────
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

fig = plt.figure(figsize=(12, 4))
for idx, azim in enumerate([0, 90, 180, 270]):
    ax = fig.add_subplot(1, 4, idx + 1, projection='3d')
    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)
    poly = Poly3DCollection(verts[faces], alpha=0.5, facecolor='lightblue', edgecolor='none')
    ax.add_collection3d(poly)
    s = np.abs(verts).max()
    ax.set_xlim(-s, s); ax.set_ylim(-s, s); ax.set_zlim(-s, s)
    ax.view_init(elev=15, azim=azim)
    ax.set_title(f'{azim}°')
    ax.axis('off')
plt.suptitle('360° Preview', fontsize=14)
plt.tight_layout()
plt.show()